In [1]:
import sys, json, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})
BAD, GOOD, NEUTRAL, ACCENT = "#c0392b", "#1e8449", "#7f8c8d", "#2c6fbb"
RES = "../data/results/"


# 00 - Statistics toolkit
Every later notebook relies on these functions, so each is checked against scipy or statsmodels (see `tests/`).

In [2]:
from abtest.analyze.tests import two_prop_ztest, welch_ttest, bootstrap_diff_ci, delta_method_ratio, srm_test
from abtest.analyze.power import sample_size_proportions, mde_proportions, days_needed
from statsmodels.stats.proportion import proportions_ztest
r = two_prop_ztest(500, 10_000, 560, 10_000)
print(f"ours: p = {r.p_value:.6f}   statsmodels: p = {proportions_ztest([560, 500], [10_000, 10_000])[1]:.6f}")
print(f"effect {r.effect:+.4f} ({r.rel_lift:+.1%}), 95% CI [{r.ci_low:+.4f}, {r.ci_high:+.4f}]")

ours: p = 0.058258   statsmodels: p = 0.058258
effect +0.0060 (+12.0%), 95% CI [-0.0002, +0.0122]


**When to use which test**

| Metric | Test |
|---|---|
| Conversion (0/1) | two-proportion z-test |
| Continuous, well behaved | Welch t-test |
| Heavy-tailed (revenue) | bootstrap, winsorising, or CUPED |
| Ratio of two per-user totals (AOV, CTR) | delta method |
| Is the split right? | chi-square sample-ratio check |

In [3]:
n = sample_size_proportions(0.05, 0.10)
print(f"To detect a +10% relative lift on a 5% baseline with 80% power you need {n:,} users per arm.")
print(f"At 12,000 users/day that is {days_needed(n, 12_000)} days; MDE with {n:,}/arm is {mde_proportions(0.05, n):.1%}.")

To detect a +10% relative lift on a 5% baseline with 80% power you need 31,231 users per arm.
At 12,000 users/day that is 6 days; MDE with 31,231/arm is 9.8%.


In [4]:
print("SRM p-value, 50,000 vs 48,500 (expected 50/50):", f"{srm_test(50_000, 48_500):.2e}", "<- flag (p < 0.001)")

SRM p-value, 50,000 vs 48,500 (expected 50/50): 1.76e-06 <- flag (p < 0.001)


**So what:** a trustworthy analysis starts with tests whose error rates are known. The Monte Carlo notebooks that follow measure exactly how those error rates break when the assumptions behind them are violated.